# YOLO26 Fine-tuning — Aerial Person Detection

Тренування YOLO26 nano на датасеті `aerial-person-detection` (7k фото, 12 класів).

**Що отримаєте:** модель, яка знаходить людей, пішоходів, авто, вантажівки та інші об'єкти на знімках з висоти (дрон, квадрокоптер, супутник).

---
### ⏱ Орієнтовний час: ~3 години на T4 (безкоштовний GPU від Google)

---
## Крок 1: Встановлення залежностей

In [ ]:
!pip install -q ultralytics roboflow

---
## Крок 2: API ключ Roboflow

Введіть ключ (отримати: app.roboflow.com → Settings → API Keys).

**Увага:** ключ не зберігається в ноутбуці, тільки в пам'яті сесії.

In [ ]:
from getpass import getpass
ROBOFLOW_API_KEY = getpass("Введіть Roboflow API ключ:")

---
## Крок 3: Завантаження датасету

Датасет: [Aerial Person Detection](https://universe.roboflow.com/aerial-person-detection/aerial-person-detection)
— 7015 фото, 12 класів (people, pedestrian, car, truck, bus, van, motor, bicycle, tricycle, awning-tricycle, ignored regions, others).

Формат: YOLOv8.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("aerial-person-detection").project("aerial-person-detection")
dataset = project.version(1).download("yolov8")

print(f"\n✅ Датасет завантажено: {dataset.location}")

---
## Крок 4: Моніторинг тренування

Ultralytics автоматично виводить прогрес у клітинці нижче після **кожної епохи**:
  `Epoch 5/150 | box_loss 0.82 | cls_loss 0.45 | mAP50 0.31...`

### Способи стежити в реальному часі:

#### 1. Вивід клітинки (найпростіше)
Після кожної епохи оновлюється рядок з loss + метриками.

#### 2. TensorBoard (найдетальніше)
Запустіть клітинку нижче **до** старту тренування — побачите графіки loss, mAP, lr у реальному часі. Оновлюється кожні 30 секунд.

#### 3. [Сповий браузер Colab (панель зліва)](https://medium.com/@jcharistech/colab-file-browser-how-to-easily-access-files-8a4e2f00eeca)
Піктограма 📁 → `runs/aerial_person/results.png` — автоматично оновлюється кожну епоху.

### Що означають метрики:
| Метрика | Добре | Пояснення |
|---------|-------|-----------|
| **mAP@0.5** | >0.50 | Точність детекції (чим вище, тим краще) |
| **box_loss** | ↓ | Похибка координат bounding box |
| **cls_loss** | ↓ | Похибка визначення класу |
| **precision** | >0.60 | Скільки з наших передбачень правильні |
| **recall** | >0.40 | Скільки справжніх об'єктів ми знайшли |

Якщо loss не знижується 30 епох поспіль — спрацює `patience=30` і тренування зупиниться.

---
## Крок 6: Запуск TensorBoard (опціонально, для моніторингу)

Запустіть цю клітинку **перед** тренуванням — відкриється панель з графіками loss, mAP, lr в реальному часі.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/aerial_person

---
## Крок 7: Тренування YOLO26 nano

Після запуску цієї клітинки тренування піде на GPU. Кожна епоха виводить:
`Epoch 1/150 | box_loss 0.95 | cls_loss 0.52 | dfl_loss 1.12 | mAP50 0.12...`

**Як стежити:** дивіться на вивід, або перемкніться на вкладку TensorBoard вище.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=150,
    imgsz=640,
    batch=32,
    device="cuda",
    patience=30,
    optimizer="AdamW",
    cos_lr=True,
    amp=True,
    project="runs",
    name="aerial_person",
    exist_ok=True,
)

---
## Крок 8: Результати

Після завершення тренування будуть виведені метрики на валідаційній вибірці:

In [ ]:
import pandas as pd
from pathlib import Path

results_path = Path("runs/aerial_person")

if results_path.exists():
    print("✅ Тренування завершено!")
    print(f"\nМодель збережено: {results_path / 'weights' / 'best.pt'}")
    print(f"Остання епоха: {results_path / 'weights' / 'last.pt'}")

    results_csv = results_path / "results.csv"
    if results_csv.exists():
        df = pd.read_csv(results_csv)
        final = df.iloc[-1]
        print("\n=== Фінальні метрики ===")
        print(f"  mAP@0.5:     {final.get('metrics/mAP_0.5', '?'):.4f}")
        print(f"  mAP@0.5:0.95:{final.get('metrics/mAP_0.5:0.95', '?'):.4f}")
        print(f"  Precision:   {final.get('metrics/precision', '?'):.4f}")
        print(f"  Recall:      {final.get('metrics/recall', '?'):.4f}")
        print(f"  Всього епох: {len(df)}")
else:
    print("❌ Щось пішло не так — папка результатів не знайдена")

---
## Крок 9: Завантаження моделі на локальну машину

Виконайте клітинку нижче — `best.pt` буде завантажено на ваш комп'ютер.

In [ ]:
from google.colab import files

best_path = "runs/aerial_person/weights/best.pt"
last_path = "runs/aerial_person/weights/last.pt"

print("Завантаження best.pt...")
files.download(best_path)

print("\n✅ Готово!")
print("\n--- Що робити далі ---")
print("1. Скопіюйте best.pt в папку detector/ вашого проекту як aerial_person.pt")
print("2. Відредагуйте detector/main.py:")
print("   - Змініть шлях до моделі:")
print('     model = AutoDetectionModel.from_pretrained(..., model_path="aerial_person.pt", ...)')
print("   - Оновіть CLASS_FILTER під назви класів датасету:")
print('     CLASS_FILTER = {')
print('         "people": ["people", "pedestrian"],')
print('         "car": ["car"],')
print('         "truck": ["truck"],')
print('         "bus": ["bus"],')
print('     }')
print("3. Перезапустіть детектор і тестуйте!")

---
## Як зрозуміти що навчання пройшло успішно

### ✅ Ознаки успіху:
- **mAP@0.5 > 0.50** — хороший результат для aerial датасету
- **mAP@0.5:0.95 > 0.30** — відмінно
- Precision i Recall стабілізувались на останніх епохах (не скачуть)
- Loss (box_loss, cls_loss, dfl_loss) плавно знижувалися без різких стрибків

### ❌ Якщо щось пішло не так:
- **mAP = 0** — неправильний формат датасету або шлях до data.yaml
- **Loss не знижується** — занадто високий lr або неправильний датасет
- **RuntimeError: CUDA out of memory** — зменшіть `batch` до 16
- **Тренування перервалось** — ноутбук зберігає `last.pt`, можна відновити:
  - Встановіть `model = YOLO('runs/aerial_person/weights/last.pt')`
  - Запустіть `train()` знову — продовжиться з останньої епохи

### 📊 Графіки:
Ultralytics автоматично генерує графіки в `runs/aerial_person/results.png` — відкрийте їх у файловому менеджері Colab (піктограма папки зліва).